---
# Fine-Tuning BERT
----

Previously in this post, I used BERT to create embeddings. I explained that fine-tuning embeddings is the best practice and the best way to create high-quality embeddings. However, fine-tuning BERT is computationally expensive and is hard to do on a CPU. 

Since classification model performance was not as good as I hoped for, I was thinking whether I could use the labelled dataset (a smaller dataset of emails) and fine-tune BERT using this along with the new email categories I added in.

## Set Up
----

In [1]:
import spacy
import numpy as np
import pandas as pd

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset

## Load Labelled Data
---

In [2]:
labelled_df = pd.read_csv('data/labelled_emails.csv')

In [3]:
labelled_df = labelled_df[labelled_df['final_label'] != 'unknown']

### Remove names, orgs and dates

In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
def remove_names(text):
    text = str(text)
    doc = nlp(text)
    for e in reversed(doc.ents):
        if e.label_ in ("PERSON", "ORG", "DATE",): 
            text = text[:e.start_char] + text[e.start_char + len(e.text):]

    return text

In [6]:
labelled_df['email'] = labelled_df['email'].apply(remove_names)

## Encode labels
---

In [7]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labelled_df['final_label'])

## Train/Test Split
---

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    labelled_df['email'],        
    labels_encoded,            
    test_size=0.2,    
    random_state=14,  
    stratify=labels_encoded # to even out classes 
)

## Prepare Datasets
---

### Tokenise Emails

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [10]:
## use previous tokeniser
def tokenise_and_chunk(email, chunk_size=512, padding_token=0):
    """
    Tokenizes the email, automatically truncates if necessary, and splits into chunks of size `chunk_size` with padding and attention mask.
    """
    # Tokenize the email, automatically truncating it to chunk_size, and adding special tokens
    tokens = tokenizer.encode(email, add_special_tokens=True, max_length=chunk_size, truncation=False)

    chunks = []
    attention_masks = []
    
    # Split tokens into chunks
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i+chunk_size]
        
        # If the chunk is smaller than chunk_size, pad it
        if len(chunk) < chunk_size:
            padding_length = chunk_size - len(chunk)
            chunk = chunk + [padding_token] * padding_length
        
        # Attention mask: 1 for real tokens, 0 for padding
        mask = [1 if token != padding_token else 0 for token in chunk]

        chunks.append(chunk)
        attention_masks.append(mask)

    return chunks, attention_masks

In [11]:
train_tokened_emails = []
train_atten_masks = []
train_labels = []

for email, label in zip(X_train, y_train):
    chunks, masks = tokenise_and_chunk(email, chunk_size=512, padding_token=0)
    train_tokened_emails.extend(chunks)
    train_atten_masks.extend(masks)
    # each chunk needs same label
    train_labels.extend([label] * len(chunks))

In [12]:
val_tokened_emails = []
val_attention_masks = []
val_labels = []
val_email_ids = []

for index, (email, label) in enumerate(zip(X_val, y_val)):
    chunks, masks = tokenise_and_chunk(email, chunk_size=512, padding_token=0)
    val_tokened_emails.extend(chunks)
    val_attention_masks.extend(masks)
    val_labels.extend([label] * len(chunks))
    # now need to track the email_id as will need to put chunks togehter later (eval)
    val_email_ids.extend([index] * len(chunks)) 



In [13]:
train_dataset = Dataset.from_dict({
    'input_ids': train_tokened_emails,
    'attention_mask': train_atten_masks,
    'labels': train_labels
})

val_dataset = Dataset.from_dict({
    'input_ids': val_tokened_emails,
    'attention_mask': val_attention_masks,
    'labels': val_labels,
    'email_id': val_email_ids
})

## Train DistilBERT Classifier
---

So we are still using DistilBERT due to limits with a CPU, but now we want model for classificaiton meaning that there is another layer on top of the BERT architecture for these labels.

The reason we fine-tune BERT for classification is that this adjusts the  embeddings to better separate the categories in the dataset. This helps the classifier layer predict the correct label and it means that the embeddings themselves can be more meaningful for tasks like clustering or similarity


### Define model

In [14]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=5 # have 5 possible categroies
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Freeze Weights

In [15]:
# freeze all weights first
for param in model.distilbert.parameters():
    param.requires_grad = False

In [16]:
# unfreeze last two layers in transformer
for layer in model.distilbert.transformer.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [17]:
# unfreeze pre_classifier and classifier layers
for param in model.pre_classifier.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

### Define Trainer and Training Args

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
)

training_args.evaluation_strategy = "epoch"


In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

### Train!

In [20]:
trainer.train()

Step,Training Loss
500,0.827000
1000,0.234700
1500,0.041500
2000,0.004900


TrainOutput(global_step=2010, training_loss=0.2756592473654605, metrics={'train_runtime': 1104.2173, 'train_samples_per_second': 14.522, 'train_steps_per_second': 1.82, 'total_flos': 2124228379161600.0, 'train_loss': 0.2756592473654605, 'epoch': 15.0})

## Evaluate Classifier
---

In [29]:
predictions = trainer.predict(val_dataset)

In [30]:
predictions

PredictionOutput(predictions=array([[-1.5183551 ,  0.49589077,  2.7399786 , -1.770364  , -1.8005503 ],
       [-2.0017622 ,  7.836157  , -3.0671537 , -3.1268158 , -2.7457528 ],
       [-3.8110526 , -0.8023125 ,  6.8925047 , -2.2652082 , -1.4657879 ],
       ...,
       [-2.4726899 , -1.3648853 , -2.8883264 ,  2.467071  ,  3.483026  ],
       [-1.9595937 , -2.604082  ,  7.907651  , -2.7359173 , -2.3065076 ],
       [ 0.86181194, -2.1306152 ,  3.5127547 , -2.8357344 , -0.6540854 ]],
      dtype=float32), label_ids=array([2, 1, 2, 4, 2, 2, 0, 0, 0, 0, 0, 3, 4, 4, 4, 3, 2, 1, 4, 0, 3, 1,
       0, 2, 0, 3, 2, 0, 2, 4, 1, 4, 3, 4, 4, 4, 4, 4, 4, 0, 3, 3, 3, 2,
       2, 4, 4, 4, 1, 3, 4, 3, 3, 1, 1, 1, 1, 1, 2, 3, 0, 0, 2, 1, 3, 1,
       1, 2, 2, 4, 3, 0, 3, 0, 1, 0, 4, 0, 0, 3, 1, 1, 1, 1, 2, 0, 4, 0,
       0, 4, 3, 1, 3, 0, 1, 0, 0, 0, 0, 2, 3, 0, 1, 1, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 1, 1, 2, 0, 0, 2, 4, 1, 3, 3, 3, 1, 1, 1, 2,
       0, 0, 0, 2, 2, 0, 4, 2, 2, 2, 0, 4, 3

In [31]:
predictions.predictions.shape

(251, 5)

In [32]:
predictions.label_ids.shape


(251,)

In [33]:
chunk_logits = predictions.predictions
chunk_preds = np.argmax(chunk_logits, axis=1)
true_labels = predictions.label_ids

In [34]:
df = pd.DataFrame({
    'email_id': val_email_ids,
    'pred_label': chunk_preds,
    'true_label': true_labels
})


In [35]:
#for each email, count the most common label
email_level_preds = df.groupby('email_id')['pred_label'].agg(lambda x: np.bincount(x).argmax())
# true labels for each email are the same for all chunks, so taking first
email_level_labels = df.groupby('email_id')['true_label'].first() 

In [36]:
acc = accuracy_score(email_level_labels, email_level_preds)
f1 = f1_score(email_level_labels, email_level_preds, average='weighted')
precision = precision_score(email_level_labels, email_level_preds, average='weighted' )
recall = recall_score(email_level_labels, email_level_preds, average='weighted')

print(f"Accuracy: {round(acc,4)}")
print(f"Precision: {round(precision,4)}")
print(f"Recall: {round(recall,4)}")
print(f"F1: {round(f1,4)}")


Accuracy: 0.6765
Precision: 0.6828
Recall: 0.6765
F1: 0.6773


## Summary
----

Overall, DistilBERT, even with fine-tuning, is not really outperforming the simpler classification models I tested. This confirmed my initial concern: there simply isn’t enough labelled data for any model to learn effectively. Since I’m not planning to spend time labelling more data, I may need to reconsider the scope of the project to better suit the data I have.

